# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [215]:
# importar librerías
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [216]:
# cargar archivos



orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')



---

In [217]:
# Ver las primeras filas
print(orders.head())

# Ver tipos de datos y nulos
print(orders.info())


# Conteo exacto de valores faltantes por columna
print(orders.isnull().sum())



  id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
0   order_0  user_6993        2025-05-22  Argentina     desktop   
1   order_1  user_1329        2025-06-15     Mexico     desktop   
2   order_2  user_3194        2025-05-02  Argentina     desktop   
3   order_3  user_4510        2025-06-09   Colombia      mobile   
4   order_4  user_5044        2025-03-30  Argentina     desktop   

  fuente_referencia       nombre_producto categoria_producto  cantidad  \
0           organic       Jacket-Winter-M               Moda       2.0   
1       paid_search  Tablet-Standard-64GB        Electronica       1.0   
2            social        Blender-XL-Red              Hogar       2.0   
3            social  Tablet-Standard-64GB        Electronica       1.0   
4       paid_search        Blender-XL-Red              Hogar       1.0   

   precio_unitario  monto_descuento  monto_total  
0           332.69              0.0       665.37  
1           176.86              5.0       171.86  

In [218]:
# Aplicamos los cambios principales
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])
orders['pais'] = orders['pais'].fillna('Unknown')
orders = orders.dropna(subset=['nombre_producto'])

print("--- VERIFICACIÓN DE NULOS ---")
print(orders.isnull().sum())
print("\n--- VERIFICACIÓN DE TIPOS (Dtypes) ---")
print(orders[['fecha_hora_pedido', 'cantidad']].dtypes)

--- VERIFICACIÓN DE NULOS ---
id_pedido              0
id_usuario             0
fecha_hora_pedido      0
pais                   0
dispositivo           20
fuente_referencia      0
nombre_producto        0
categoria_producto    50
cantidad              50
precio_unitario       50
monto_descuento       50
monto_total            0
dtype: int64

--- VERIFICACIÓN DE TIPOS (Dtypes) ---
fecha_hora_pedido    datetime64[ns]
cantidad                    float64
dtype: object


In [219]:
# Recalculamos para validar
orders['monto_total_validado'] = (orders['cantidad'] * orders['precio_unitario']) - orders['monto_descuento']

print("\n--- COMPARACIÓN DE MONTOS (Primeras 5 filas) ---")
print(orders[['id_pedido', 'monto_total', 'monto_total_validado']].head())

# Chequeo rápido de errores de cálculo
errores = orders[abs(orders['monto_total'] - orders['monto_total_validado']) > 0.01]
print(f"\nCantidad de registros con discrepancias: {len(errores)}")


--- COMPARACIÓN DE MONTOS (Primeras 5 filas) ---
  id_pedido  monto_total  monto_total_validado
0   order_0       665.37                665.38
1   order_1       171.86                171.86
2   order_2       195.99                195.98
3   order_3       242.87                242.87
4   order_4       336.28                336.28

Cantidad de registros con discrepancias: 1149


In [220]:
# 1. Reemplazamos el monto original por el validado
orders['monto_total'] = orders['monto_total_validado']

# 2. Eliminamos la columna auxiliar para que el dataset quede limpio
orders = orders.drop(columns=['monto_total_validado'])

print("✅ Montos corregidos y estandarizados.")
print(orders[['id_pedido', 'monto_total']].head())

✅ Montos corregidos y estandarizados.
  id_pedido  monto_total
0   order_0       665.38
1   order_1       171.86
2   order_2       195.98
3   order_3       242.87
4   order_4       336.28


In [221]:
# Cargar el dataset
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')

# Revisar estructura y nulos
print("--- INFO DEL CATÁLOGO ---")
print(catalog.info())
print("\n--- VALORES NULOS ---")
print(catalog.isnull().sum())

--- INFO DEL CATÁLOGO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes
None

--- VALORES NULOS ---
nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64


In [222]:
# 1. Quitar espacios en blanco extra y estandarizar mayúsculas
catalog['nombre_producto'] = catalog['nombre_producto'].str.strip().str.title()
catalog['categoria_producto'] = catalog['categoria_producto'].str.strip().str.title()
catalog['proveedor'] = catalog['proveedor'].str.strip().str.title()

# 2. Verificación de duplicados (por si acaso el ID del producto se repite)
catalog = catalog.drop_duplicates(subset=['nombre_producto'])

print("✅ Texto estandarizado y duplicados eliminados.")

✅ Texto estandarizado y duplicados eliminados.


In [223]:
print("--- RESUMEN DE COSTOS ---")
print(catalog['costo_unitario'].describe())

--- RESUMEN DE COSTOS ---
count      7.000000
mean     102.252857
std      111.011563
min       10.120000
25%       16.905000
50%       25.210000
75%      182.975000
max      280.680000
Name: costo_unitario, dtype: float64


In [224]:
# 1. Corregir "Mexico" para que tenga acento como en el dataset de orders
marketing['pais'] = marketing['pais'].replace('Mexico', 'México')

# 2. Asegurarnos que los canales estén en minúsculas (igual que en orders)
marketing['canal'] = marketing['canal'].str.lower().str.strip()

print("✅ Marketing estandarizado. Ahora 'México' tiene acento.")

✅ Marketing estandarizado. Ahora 'México' tiene acento.


In [225]:
# Unimos la tabla de órdenes con la del catálogo
orders_full = orders.merge(catalog[['nombre_producto', 'costo_unitario', 'proveedor']], 
                           on='nombre_producto', 
                           how='left')

In [226]:
# Muestra las primeras 10 filas con un formato más limpio
display(orders_full.head(10))

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,costo_unitario,proveedor
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.38,189.31,Mcmillan-Rhodes
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86,NaN,NaN
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.98,NaN,NaN
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87,NaN,NaN
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28,NaN,NaN
5,order_5,user_2792,2025-04-22,mexico,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,179.34,0.0,179.34,NaN,NaN
6,order_6,user_2802,2025-03-15,Colombia,mobile,organic,Blender-XL-Red,Hogar,2.0,163.59,0.0,327.18,NaN,NaN
7,order_7,user_1083,2025-05-01,Mexico,desktop,organic,Tablet-Standard-64GB,Electronica,2.0,373.68,0.0,747.36,NaN,NaN
8,order_8,user_2352,2025-03-01,Argentina,desktop,paid_search,Laptop-Gaming-16GB,Electronica,2.0,477.27,5.0,949.54,NaN,NaN
9,order_9,user_7001,2025-05-31,Mexico,mobile,organic,Sneakers-Urban-42,Moda,1.0,339.30,5.0,334.30,17.21,Greene-Smith


In [227]:
# Limpiamos los nombres en ambas tablas para que el "match" sea total
orders['nombre_producto'] = orders['nombre_producto'].str.strip()
catalog['nombre_producto'] = catalog['nombre_producto'].str.strip()

# Rehacemos el merge
orders_full = orders.merge(catalog[['nombre_producto', 'costo_unitario', 'proveedor']], 
                           on='nombre_producto', 
                           how='left')

# Verificamos si bajaron los nulos
print(f"Nulos en costo tras la limpieza: {orders_full['costo_unitario'].isnull().sum()}")


Nulos en costo tras la limpieza: 12519


In [228]:
# 1. Limpieza profunda en la tabla de Órdenes
orders['nombre_producto'] = orders['nombre_producto'].str.strip().str.lower()

# 2. Limpieza profunda idéntica en la tabla de Catálogo
catalog['nombre_producto'] = catalog['nombre_producto'].str.strip().str.lower()

# 3. Re-intentar el Merge
orders_full = orders.merge(catalog[['nombre_producto', 'costo_unitario', 'proveedor']], 
                           on='nombre_producto', 
                           how='left')

# 4. Verificar nuevamente
print(f"Nulos en costo después de la limpieza profunda: {orders_full['costo_unitario'].isnull().sum()}")


Nulos en costo después de la limpieza profunda: 0


In [229]:
# 1. Quitamos espacios invisibles para que los nombres coincidan perfecto
orders['nombre_producto'] = orders['nombre_producto'].astype(str).str.strip()
catalog['nombre_producto'] = catalog['nombre_producto'].astype(str).str.strip()

# 2. Hacemos la unión (el merge que sí necesitas)
# Usamos 'left' para mantener todas tus órdenes
orders_full = orders.merge(catalog, on='nombre_producto', how='left')

# 3. ELIMINAMOS las filas que no encontraron precio (los espacios vacíos)
# Esto limpia lo que viste en la captura de Excel
orders_full = orders_full.dropna(subset=['precio_unitario'])

print(f"¡Listo! Ahora tienes {len(orders_full)} filas limpias y completas.")

¡Listo! Ahora tienes 25020 filas limpias y completas.


### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [230]:
# --- VALIDACIÓN Y CONVERSIÓN DE FECHAS ---

# 1. Convertir la columna de órdenes (que tiene hora) a formato datetime
orders_full['fecha_hora_pedido'] = pd.to_datetime(orders_full['fecha_hora_pedido'])

# 2. Crear una columna de solo fecha (normalizada) para facilitar el análisis
# Esto elimina la hora y deja solo Año-Mes-Día
orders_full['fecha'] = orders_full['fecha_hora_pedido'].dt.normalize()

# 3. Convertir la columna de fecha en el dataset de marketing
marketing['fecha'] = pd.to_datetime(marketing['fecha'])

# 4. Verificación final de formatos
print("--- FORMATOS DE COLUMNAS ---")
print(f"Órdenes (fecha_hora): {orders_full['fecha_hora_pedido'].dtype}")
print(f"Órdenes (fecha):      {orders_full['fecha'].dtype}")
print(f"Marketing (fecha):    {marketing['fecha'].dtype}")

--- FORMATOS DE COLUMNAS ---
Órdenes (fecha_hora): datetime64[ns]
Órdenes (fecha):      datetime64[ns]
Marketing (fecha):    datetime64[ns]


In [231]:
# Definimos las columnas numéricas clave
cols_validar = ['cantidad', 'precio_unitario', 'monto_total', 'costo_unitario']

print("--- REVISIÓN DE VALORES MÍNIMOS ---")
for col in cols_validar:
    min_val = orders_full[col].min()
    print(f"Valor mínimo en {col}: {min_val}")
    
    # Contar cuántos registros son negativos o cero
    invalidos = (orders_full[col] <= 0).sum()
    if invalidos > 0:
        print(f"   ⚠️ ¡Alerta! Se encontraron {invalidos} registros con valores <= 0")

--- REVISIÓN DE VALORES MÍNIMOS ---
Valor mínimo en cantidad: -2.0
   ⚠️ ¡Alerta! Se encontraron 4 registros con valores <= 0
Valor mínimo en precio_unitario: 20.03
Valor mínimo en monto_total: -502.65
   ⚠️ ¡Alerta! Se encontraron 4 registros con valores <= 0
Valor mínimo en costo_unitario: 10.12


In [232]:
# Creamos una columna temporal para calcular el monto esperado
orders_full['monto_esperado'] = (orders_full['precio_unitario'] * orders_full['cantidad']) - orders_full['monto_descuento']

# Calculamos la diferencia entre lo registrado y lo calculado
# Usamos round(2) para evitar errores por decimales mínimos
diferencia = (orders_full['monto_total'].round(2) - orders_full['monto_esperado'].round(2)).abs().sum()

if diferencia == 0:
    print("✅ ¡Consistencia total! Los montos totales coinciden perfectamente con precio, cantidad y descuento.")
else:
    print(f"⚠️ Atención: Hay una diferencia acumulada de {diferencia}. Revisando filas con error...")
    # Ver filas donde la diferencia es mayor a 0.01 centavos
    errores = orders_full[(orders_full['monto_total'].round(2) - orders_full['monto_esperado'].round(2)).abs() > 0.01]
    display(errores.head())

✅ ¡Consistencia total! Los montos totales coinciden perfectamente con precio, cantidad y descuento.


In [233]:
# Contar filas duplicadas totales en el dataset
duplicados_totales = orders_full.duplicated().sum()

# Contar si hay IDs de pedido repetidos (lo más crítico)
ids_duplicados = orders_full['id_pedido'].duplicated().sum()

print(f"Filas exactamente iguales: {duplicados_totales}")
print(f"IDs de pedido repetidos: {ids_duplicados}")

Filas exactamente iguales: 100
IDs de pedido repetidos: 100


In [234]:
# 1. Volvemos a crear orders_full 
orders_full = orders.merge(catalog[['nombre_producto', 'costo_unitario', 'proveedor']], 
                           on='nombre_producto', 
                           how='left')

# 2. ELIMINACIÓN DE DUPLICADOS 
# Esto borra las filas idénticas y deja solo una versión de cada pedido
orders_full = orders_full.drop_duplicates(keep='first')

# 3. Aseguramos el formato de fechas (Validación final)
orders_full['fecha'] = pd.to_datetime(orders_full['fecha_hora_pedido']).dt.normalize()
marketing['fecha'] = pd.to_datetime(marketing['fecha'])

# 4. Verificación final de limpieza
print(f"✅ Duplicados eliminados. Filas actuales: {len(orders_full)}")
print(f"✅ Nulos en costo: {orders_full['costo_unitario'].isnull().sum()}")
print(f"✅ Formato de fecha: {orders_full['fecha'].dtype}")

# Ver las columnas para que no te vuelva a marcar error
print("\nColumnas listas:", orders_full.columns.tolist())

✅ Duplicados eliminados. Filas actuales: 24970
✅ Nulos en costo: 0
✅ Formato de fecha: datetime64[ns]

Columnas listas: ['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto', 'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total', 'costo_unitario', 'proveedor', 'fecha']


In [235]:
# Columnas a revisar
categoricas = ['pais', 'dispositivo', 'fuente_referencia', 'categoria_producto']

print("--- REVISIÓN DE CATEGORÍAS ÚNICAS ---")
for col in categoricas:
    valores = orders_full[col].unique()
    print(f"\nValores en {col}:")
    print(valores)

--- REVISIÓN DE CATEGORÍAS ÚNICAS ---

Valores en pais:
['Argentina' 'Mexico' 'Colombia' 'mexico' 'colombia' 'argentina' 'Unknown']

Valores en dispositivo:
['desktop' 'mobile' nan]

Valores en fuente_referencia:
['organic' 'paid_search' 'social']

Valores en categoria_producto:
['Moda' 'Electronica' 'Hogar' nan]


In [236]:
# 1. Estandarizamos los valores desconocidos a un solo idioma o formato
orders_full['pais'] = orders_full['pais'].replace('Unknown', 'No Identificado')
orders_full['dispositivo'] = orders_full['dispositivo'].replace('Desconocido', 'no identificado')

# 2. Pasamos todo a minúsculas en 'dispositivo' para que sea uniforme con 'pais'
orders_full['dispositivo'] = orders_full['dispositivo'].str.lower()

print("✅ Categorías estandarizadas.")
print(f"Países únicos ahora: {orders_full['pais'].unique()}")

✅ Categorías estandarizadas.
Países únicos ahora: ['Argentina' 'Mexico' 'Colombia' 'mexico' 'colombia' 'argentina'
 'No Identificado']


In [237]:
# 1. Aseguramos que la columna sea tipo datetime
orders_full['fecha'] = pd.to_datetime(orders_full['fecha'])

# 2. Convertimos la columna a solo fecha (formato YYYY-MM-DD)
orders_full['fecha'] = orders_full['fecha'].dt.date

# 3. Ahora sí, exportamos de nuevo
orders_full.to_excel('reporte_final_limpio.xlsx', index=False)

print("¡Reporte actualizado! Ahora las fechas se verán sin la hora.")

¡Reporte actualizado! Ahora las fechas se verán sin la hora.


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [238]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [239]:
# Calculamos el ingreso total sumando la columna monto_total
ingreso_total = orders_full['monto_total'].sum()

# Lo imprimimos con formato de moneda para que se vea profesional
print(f"El ingreso total (Revenue) es: ${ingreso_total:,.2f}")

El ingreso total (Revenue) es: $51,953,532.48


In [240]:
# Calculamos el costo total por cada fila (Costo Unitario * Cantidad)
# Nota: Es importante usar la cantidad porque no siempre se vende de a 1 unidad
orders_full['costo_total_pedido'] = orders_full['costo_unitario'] * orders_full['cantidad']

# Sumamos todo para obtener el Costo Total Global
costo_total_global = orders_full['costo_total_pedido'].sum()

print(f"El costo total de los productos vendidos es: ${costo_total_global:,.2f}")

El costo total de los productos vendidos es: $43,124,018.41


In [241]:
# Costo total repartido por país
costo_por_pais = orders_full.groupby('pais')['costo_total_pedido'].sum().sort_values(ascending=False)

print("--- Costo Total por País ---")
print(costo_por_pais.map('${:,.2f}'.format))

--- Costo Total por País ---
pais
Mexico             $15,185,979.15
Argentina          $15,155,476.92
Colombia            $9,566,740.64
mexico              $2,933,318.76
colombia              $124,289.50
argentina             $114,554.30
No Identificado        $43,659.14
Name: costo_total_pedido, dtype: object


In [242]:
# 1. Limpiamos los nombres de las columnas (quita espacios y pone minúsculas)
marketing.columns = marketing.columns.str.strip().str.lower()

# 2. Ahora intentamos sumar de nuevo
inversion_marketing_total = marketing['gasto'].sum()

print(f"La inversión total en marketing es: ${inversion_marketing_total:,.2f}")

La inversión total en marketing es: $2,871,843.53


In [243]:
# 1. Definimos de nuevo los totales (asegúrate de haber corrido las celdas anteriores de sumas)
# Si te marca error en alguna de estas, es que el nombre de la variable cambió
ingreso_total = orders_full['monto_total'].sum()
costo_total_global = (orders_full['costo_unitario'] * orders_full['cantidad']).sum()
inversion_marketing_total = marketing['gasto'].sum()

# 2. Calculamos la Ganancia Bruta (Gross Profit)
ganancia_bruta_total = ingreso_total - costo_total_global

# 3. Calculamos la Utilidad Neta (Net Profit)
utilidad_neta = ganancia_bruta_total - inversion_marketing_total

# 4. Calculamos el margen neto
margen_neto_pct = (utilidad_neta / ingreso_total) * 100

print(f"--- VERDICTO DE RENTABILIDAD ---")
print(f"Ingresos Totales:      ${ingreso_total:,.2f}")
print(f"Ganancia Bruta:        ${ganancia_bruta_total:,.2f}")
print(f"Utilidad Neta (Profit): ${utilidad_neta:,.2f}")
print(f"Margen Neto:            {margen_neto_pct:.2f}%")

if utilidad_neta > 0:
    print("\n✅ EL NEGOCIO ES RENTABLE")
else:
    print("\n❌ EL NEGOCIO NO ES RENTABLE")

--- VERDICTO DE RENTABILIDAD ---
Ingresos Totales:      $51,953,532.48
Ganancia Bruta:        $8,829,514.07
Utilidad Neta (Profit): $5,957,670.54
Margen Neto:            11.47%

✅ EL NEGOCIO ES RENTABLE


In [244]:
# El ticket promedio es la suma del monto_total entre el conteo de pedidos únicos
ticket_promedio = orders_full['monto_total'].mean()

# También se puede calcular así para mayor claridad:
# ticket_promedio = orders_full['monto_total'].sum() / orders_full['id_pedido'].nunique()

print(f"El Ticket Promedio por orden es: ${ticket_promedio:,.2f}")

El Ticket Promedio por orden es: $2,084.81


In [245]:
# Agrupamos por país y calculamos el promedio del monto_total
aov_por_pais = orders_full.groupby('pais')['monto_total'].mean().sort_values(ascending=False)

print("--- Ticket Promedio por País ---")
print(aov_por_pais.map('${:,.2f}'.format))

--- Ticket Promedio por País ---
pais
Argentina          $2,821.76
Mexico             $2,490.44
Colombia           $1,481.19
mexico             $1,389.89
argentina            $400.29
colombia             $388.25
No Identificado      $369.34
Name: monto_total, dtype: object


In [246]:
# Calculamos el promedio de la columna cantidad
cantidad_promedio = orders_full['cantidad'].mean()

print(f"La cantidad promedio de productos por orden es: {cantidad_promedio:.2f} artículos")

La cantidad promedio de productos por orden es: 7.12 artículos


In [247]:
# Promedio de productos por país
cantidad_por_pais = orders_full.groupby('pais')['cantidad'].mean().sort_values(ascending=False)

print("--- Cantidad Promedio de Productos por País ---")
print(cantidad_por_pais.round(2))

--- Cantidad Promedio de Productos por País ---
pais
mexico             13.08
Argentina           8.42
Mexico              8.22
Colombia            5.52
argentina           1.52
colombia            1.50
No Identificado     1.46
Name: cantidad, dtype: float64


In [248]:
# Agrupamos por nombre de producto y sumamos la cantidad
top_productos = orders_full.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False).head(5)

print("--- TOP 5 PRODUCTOS POR CANTIDAD VENDIDA ---")
print(top_productos)

--- TOP 5 PRODUCTOS POR CANTIDAD VENDIDA ---
nombre_producto
laptop-gaming-16gb    144198.0
vacuum-pro-black        6284.0
blender-xl-red          6279.0
jacket-winter-m         6256.0
sneakers-urban-42       6172.0
Name: cantidad, dtype: float64


In [249]:
# Agrupamos por categoría y sumamos la cantidad
ventas_por_categoria = orders_full.groupby('categoria_producto')['cantidad'].sum().sort_values(ascending=False)

# Calculamos el porcentaje para el reporte
pct_categoria = (ventas_por_categoria / ventas_por_categoria.sum()) * 100

print("\n--- DISTRIBUCIÓN POR CATEGORÍA ---")
for cat, cant in ventas_por_categoria.items():
    print(f"{cat}: {cant} unidades ({pct_categoria[cat]:.2f}%)")


--- DISTRIBUCIÓN POR CATEGORÍA ---
Electronica: 152486.0 unidades (85.92%)
Hogar: 12563.0 unidades (7.08%)
Moda: 12428.0 unidades (7.00%)


In [250]:
# Agrupamos por canal y sumamos la columna 'gasto'
gasto_por_canal = marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)

# Lo mostramos con formato de moneda
print("--- INVERSIÓN TOTAL POR CANAL ---")
print(gasto_por_canal.map('${:,.2f}'.format))

# Calculamos el porcentaje de participación
participacion_gasto = (gasto_por_canal / gasto_por_canal.sum()) * 100
print("\n--- PORCENTAJE DE DISTRIBUCIÓN ---")
print(participacion_gasto.map('{:.2f}%'.format))

--- INVERSIÓN TOTAL POR CANAL ---
canal
social         $918,043.21
organic        $913,533.01
paid_search    $863,088.21
Name: gasto, dtype: object

--- PORCENTAJE DE DISTRIBUCIÓN ---
canal
social         34.07%
organic        33.90%
paid_search    32.03%
Name: gasto, dtype: object


In [251]:
# Verificamos el tipo de dato original de la columna gasto antes del formato
print(marketing['gasto'].dtype)

float64


## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [252]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [253]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [254]:
# PARTE 1: Totales del funnel
# ======================

query_totals ='''
SELECT 
nombre_evento, 
COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM  events
GROUP BY nombre_evento
ORDER BY usuarios_unicos DESC;
'''
totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [255]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel_totals AS (
    SELECT 
        nombre_evento, 
        COUNT(DISTINCT id_usuario) AS usuarios_unicos
    FROM 
        events
    GROUP BY 
        nombre_evento
)
SELECT 
    nombre_evento,
    usuarios_unicos,
    ROUND(usuarios_unicos * 100.0 / FIRST_VALUE(usuarios_unicos) OVER (ORDER BY usuarios_unicos DESC), 2) AS porcentaje_total,
    ROUND(usuarios_unicos * 100.0 / LAG(usuarios_unicos) OVER (ORDER BY usuarios_unicos DESC), 2) AS porcentaje_paso_anterior
FROM 
    funnel_totals
ORDER BY 
    usuarios_unicos DESC;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios_unicos,porcentaje_total,porcentaje_paso_anterior
0,first_visit,7796,100.00,NaN
1,add_to_cart,7634,97.92,97.92
2,select_item,7582,97.26,99.32
3,begin_checkout,7208,92.46,95.07
4,add_payment_info,6250,80.17,86.71
5,purchase,6240,80.04,99.84


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [256]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [257]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT 
    id_usuario, 
    CAST(fecha_actividad AS DATE) AS fecha_actividad, 
    dias_despues_registro, 
    activo
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [258]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (

SELECT
    id_usuario,
    DATE_TRUNC(
        'month',
        CAST(fecha_registro AS DATE)
    ) AS cohorte
FROM users

),

retencion AS (

SELECT
    c.cohorte,
    ua.id_usuario,

    MAX(
        CASE
            WHEN ua.dias_despues_registro BETWEEN 1 AND 7
            AND ua.activo = 1
            THEN 1 ELSE 0
        END
    ) AS retenido_w1,

    MAX(
        CASE
            WHEN ua.dias_despues_registro BETWEEN 8 AND 14
            AND ua.activo = 1
            THEN 1 ELSE 0
        END
    ) AS retenido_w2,

    MAX(
        CASE
            WHEN ua.dias_despues_registro BETWEEN 15 AND 21
            AND ua.activo = 1
            THEN 1 ELSE 0
        END
    ) AS retenido_w3

FROM cohortes c
LEFT JOIN user_activity ua
ON c.id_usuario = ua.id_usuario

GROUP BY
    c.cohorte,
    ua.id_usuario

)

SELECT
    cohorte,

    COUNT(id_usuario) AS usuarios,

    SUM(retenido_w1) AS retenido_w1,
    SUM(retenido_w2) AS retenido_w2,
    SUM(retenido_w3) AS retenido_w3,

    ROUND(
        SUM(retenido_w1)::numeric /
        COUNT(id_usuario) * 100,
        2
    ) AS semana_1,

    ROUND(
        SUM(retenido_w2)::numeric /
        COUNT(id_usuario) * 100,
        2
    ) AS semana_2,

    ROUND(
        SUM(retenido_w3)::numeric /
        COUNT(id_usuario) * 100,
        2
    ) AS semana_3

FROM retencion

GROUP BY cohorte

ORDER BY cohorte;

'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,usuarios,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.20,40.07,41.85


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** ...
   - **H₁ (Hipótesis alternativa):** ...
   
**Test estadístico:** ...  
**Nivel de significancia alpha:** ...

In [259]:
# tu código aquí
import pandas as pd
from scipy import stats


In [260]:
# Cargar el archivo
df_exp = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')

In [261]:
# 2. Separar grupos por la columna 'variante'
control = df_exp[df_exp['variante'] == 'control']['convirtio']
tratamiento = df_exp[df_exp['variante'] == 'tratamiento']['convirtio']


---

In [262]:
# 3. Ejecutar Test Estadístico
t_stat, p_value = stats.ttest_ind(control, tratamiento)


In [263]:
# 4. Mostrar resultados
print(f"Tasa Conversión Control: {control.mean():.2%}")
print(f"Tasa Conversión Tratamiento: {tratamiento.mean():.2%}")
print(f"P-valor: {p_value:.4f}")


Tasa Conversión Control: 15.69%
Tasa Conversión Tratamiento: 16.29%
P-valor: 0.4161


In [264]:

alpha = 0.05

print("-" * 30)
print(" RESULTADOS DEL TEST A/B ")
print("-" * 30)
print(f"Estadístico T: {t_stat:.4f}")
print(f"P-valor:       {p_value:.4f}")
print(f"Nivel Alpha:   {alpha}")
print("-" * 30)

if p_value < alpha:
    print("CONCLUSIÓN: Rechazamos la Hipótesis Nula (H0).")
    print("La diferencia es ESTADÍSTICAMENTE SIGNIFICATIVA.")
    print("Recomendación: Implementar la nueva interfaz (UI).")
else:
    print("CONCLUSIÓN: No podemos rechazar la Hipótesis Nula (H0).")
    print("La diferencia NO es estadísticamente significativa.")
    print("Recomendación: Mantener la interfaz actual.")
print("-" * 30)

------------------------------
 RESULTADOS DEL TEST A/B 
------------------------------
Estadístico T: -0.8132
P-valor:       0.4161
Nivel Alpha:   0.05
------------------------------
CONCLUSIÓN: No podemos rechazar la Hipótesis Nula (H0).
La diferencia NO es estadísticamente significativa.
Recomendación: Mantener la interfaz actual.
------------------------------


In [265]:
orders_full.to_excel('reporte_final.xlsx', index=False)

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [266]:
# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive